# Landslide step 07: baseline EAD min/max comparison maps

Compares **baseline total EAD (USD)** between minimum and maximum scenarios using side-by-side maps with a shared color scale.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable


In [ ]:
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
network_csv = base_path / 'dphil_common_cross_cutting/common_incoming_data/networks/network_layers_hazard_intersections_details.csv'
jamaica_boundary_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'
data_root = base_path / 'dphil_common_cross_cutting/common_incoming_data'

min_asset_ead_csv = base_path / 'dphil_paper_3/results/02_damage_estimates/landslide_damages/results_landslide_minimum_scenario/damage_estimates/landslide_ead_asset_level_usd.csv'
max_asset_ead_csv = base_path / 'dphil_paper_3/results/02_damage_estimates/landslide_damages/results_landslide_maximum_scenario/damage_estimates/landslide_ead_asset_level_usd.csv'

comparison_out_dir = base_path / 'dphil_paper_3/results/02_damage_estimates/landslide_damages/min_max_comparison_maps'
comparison_out_dir.mkdir(parents=True, exist_ok=True)

for required in [network_csv, jamaica_boundary_path, min_asset_ead_csv, max_asset_ead_csv]:
    if not required.exists():
        raise FileNotFoundError(f'Missing required file: {required}')

print('Minimum scenario asset EAD:', min_asset_ead_csv)
print('Maximum scenario asset EAD:', max_asset_ead_csv)
print('Output directory:', comparison_out_dir)


# Performance controls
save_map_layers = False  # set True if you want to export full asset map layers (can be very large)
max_plot_features = 250000  # random sample cap for plotting speed


In [ ]:
network_details = pd.read_csv(network_csv)[[
    'sector', 'asset_description', 'asset_gpkg', 'asset_layer', 'asset_id_column', 'path'
]].drop_duplicates()


def resolve_network_asset_file(asset_relative_path):
    relative_asset_path = Path(asset_relative_path)
    path_a = data_root / relative_asset_path
    path_b = data_root / 'networks' / relative_asset_path

    if path_a.exists():
        return path_a
    if path_b.exists():
        return path_b

    raise FileNotFoundError(
        f"Could not find asset file '{relative_asset_path}'. Checked: {path_a} ; {path_b}"
    )


def build_baseline_map_layer(asset_ead_csv: Path, scenario_label: str) -> gpd.GeoDataFrame:
    asset_ead = pd.read_csv(asset_ead_csv, low_memory=False)
    required = ['Asset', 'Layer', 'Asset_ID', 'EAD_Baseline_USD']
    missing = [c for c in required if c not in asset_ead.columns]
    if missing:
        raise KeyError(f'{asset_ead_csv.name} missing columns: {missing}')

    map_layers = []

    for row in network_details.itertuples(index=False):
        subset = asset_ead.loc[
            (asset_ead['Asset'] == row.asset_gpkg) & (asset_ead['Layer'] == row.asset_layer),
            ['Asset_ID', 'EAD_Baseline_USD']
        ].copy()
        if subset.empty:
            continue

        asset_file = resolve_network_asset_file(row.path)
        asset_gdf = gpd.read_file(asset_file, layer=row.asset_layer)
        if row.asset_id_column not in asset_gdf.columns:
            continue

        asset_gdf = asset_gdf[[row.asset_id_column, 'geometry']].copy()
        asset_gdf = gpd.GeoDataFrame(asset_gdf, geometry='geometry', crs=asset_gdf.crs)
        if asset_gdf.crs is not None:
            asset_gdf = asset_gdf.to_crs('EPSG:3448')
        else:
            asset_gdf = asset_gdf.set_crs('EPSG:3448', allow_override=True)

        asset_gdf['_join_id'] = asset_gdf[row.asset_id_column].astype(str)
        subset['_join_id'] = subset['Asset_ID'].astype(str)

        merged = asset_gdf.merge(subset, on='_join_id', how='inner')
        if merged.empty:
            continue

        merged['Scenario'] = scenario_label
        merged['Sector'] = row.sector
        merged['Subsector'] = row.asset_description
        merged['Asset'] = row.asset_gpkg
        merged['Layer'] = row.asset_layer

        map_layers.append(merged[[
            'Scenario', 'Sector', 'Subsector', 'Asset', 'Layer',
            row.asset_id_column, 'Asset_ID', 'EAD_Baseline_USD', 'geometry'
        ]].rename(columns={row.asset_id_column: 'Asset_ID_Source'}))

    if not map_layers:
        raise ValueError(f'No map layers built for {scenario_label}')

    return gpd.GeoDataFrame(pd.concat(map_layers, ignore_index=True), geometry='geometry', crs='EPSG:3448')


In [ ]:
baseline_min_gdf = build_baseline_map_layer(min_asset_ead_csv, 'minimum')
baseline_max_gdf = build_baseline_map_layer(max_asset_ead_csv, 'maximum')

print('Minimum map features:', len(baseline_min_gdf))
print('Maximum map features:', len(baseline_max_gdf))

baseline_min_gdf[['EAD_Baseline_USD']].describe(percentiles=[0.5, 0.9, 0.99])


In [ ]:
# Optional: save full map layers for reuse (can be large, especially with building polygons)
if save_map_layers:
    min_layer_out = comparison_out_dir / 'baseline_ead_min_asset_map_layers.gpkg'
    max_layer_out = comparison_out_dir / 'baseline_ead_max_asset_map_layers.gpkg'

    baseline_min_gdf.to_file(min_layer_out, driver='GPKG')
    baseline_max_gdf.to_file(max_layer_out, driver='GPKG')

    print('Saved:', min_layer_out)
    print('Saved:', max_layer_out)
else:
    print('Skipped saving full GPKG map layers (save_map_layers=False).')


In [ ]:
jamaica_boundary = gpd.read_file(jamaica_boundary_path).to_crs('EPSG:3448')

all_vals = pd.concat([
    baseline_min_gdf['EAD_Baseline_USD'].astype(float),
    baseline_max_gdf['EAD_Baseline_USD'].astype(float)
], ignore_index=True).fillna(0.0)

# Robust shared color scale for clear comparison
display_quantile = 0.995
max_usd = float(all_vals.quantile(display_quantile))
if max_usd <= 0:
    max_usd = float(all_vals.max()) if float(all_vals.max()) > 0 else 1.0

if max_usd >= 1e6:
    unit_factor = 1e6
    unit_label = 'USD millions'
elif max_usd >= 1e3:
    unit_factor = 1e3
    unit_label = 'USD thousands'
else:
    unit_factor = 1.0
    unit_label = 'USD'

vmax = max_usd / unit_factor
norm = Normalize(vmin=0, vmax=vmax)
cmap = plt.cm.viridis


def sample_for_plot(gdf, max_features):
    if len(gdf) <= max_features:
        return gdf
    return gdf.sample(n=max_features, random_state=42)


def plot_baseline_panel(ax, gdf, title):
    plot_gdf = sample_for_plot(gdf.copy(), max_plot_features)
    if len(plot_gdf) < len(gdf):
        print(f"{title}: plotting sample {len(plot_gdf):,} / {len(gdf):,} features")

    plot_gdf['_plot_val'] = (plot_gdf['EAD_Baseline_USD'] / unit_factor).clip(0, vmax)

    ax.set_facecolor('#ffffff')
    jamaica_boundary.boundary.plot(ax=ax, color='#bdbdbd', linewidth=0.45, zorder=1)

    geom_type = plot_gdf.geometry.geom_type.astype(str)
    polys = plot_gdf[geom_type.str.contains('Polygon', na=False)]
    lines = plot_gdf[geom_type.str.contains('LineString', na=False)]
    points = plot_gdf[geom_type.str.contains('Point', na=False)]

    if not polys.empty:
        polys.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, linewidth=0.10, edgecolor='none', alpha=0.9, zorder=2)
    if not lines.empty:
        lines.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, linewidth=0.85, alpha=0.95, zorder=3)
    if not points.empty:
        points.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, markersize=14, alpha=0.95, zorder=4)

    ax.set_title(title, fontsize=12)
    ax.set_axis_off()


fig, axes = plt.subplots(1, 2, figsize=(16, 8), constrained_layout=True)
plot_baseline_panel(axes[0], baseline_min_gdf, 'Baseline EAD - Minimum scenario')
plot_baseline_panel(axes[1], baseline_max_gdf, 'Baseline EAD - Maximum scenario')

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), fraction=0.03, pad=0.02)
cbar.set_label(f'Baseline EAD ({unit_label}), clipped at q={display_quantile:.3f}')

out_png = comparison_out_dir / 'baseline_ead_min_max_side_by_side_shared_scale.png'
fig.savefig(out_png, dpi=300, bbox_inches='tight')
print('Saved:', out_png)
plt.show()


In [ ]:
# Difference map (maximum - minimum) at asset level
# Positive means max scenario gives higher baseline EAD than min scenario.

min_key = baseline_min_gdf[['Asset', 'Layer', 'Asset_ID', 'EAD_Baseline_USD', 'geometry']].copy()
max_key = baseline_max_gdf[['Asset', 'Layer', 'Asset_ID', 'EAD_Baseline_USD']].copy()

min_key['_k'] = min_key['Asset'].astype(str) + '|' + min_key['Layer'].astype(str) + '|' + min_key['Asset_ID'].astype(str)
max_key['_k'] = max_key['Asset'].astype(str) + '|' + max_key['Layer'].astype(str) + '|' + max_key['Asset_ID'].astype(str)

joined = min_key.merge(
    max_key[['_k', 'EAD_Baseline_USD']],
    on='_k',
    how='inner',
    suffixes=('_min', '_max')
)

joined['Baseline_EAD_Diff_MaxMinusMin_USD'] = joined['EAD_Baseline_USD_max'] - joined['EAD_Baseline_USD_min']

diff_gdf = gpd.GeoDataFrame(joined, geometry='geometry', crs='EPSG:3448')

vals = diff_gdf['Baseline_EAD_Diff_MaxMinusMin_USD'].fillna(0.0)
cap_usd = float(vals.abs().quantile(0.995))
if cap_usd <= 0:
    cap_usd = float(vals.abs().max()) if float(vals.abs().max()) > 0 else 1.0

if cap_usd >= 1e6:
    diff_factor = 1e6
    diff_unit = 'USD millions'
elif cap_usd >= 1e3:
    diff_factor = 1e3
    diff_unit = 'USD thousands'
else:
    diff_factor = 1.0
    diff_unit = 'USD'

cap = cap_usd / diff_factor
diff_gdf['_plot_val'] = (vals / diff_factor).clip(-cap, cap)

diff_plot_gdf = diff_gdf if len(diff_gdf) <= max_plot_features else diff_gdf.sample(n=max_plot_features, random_state=42)
if len(diff_plot_gdf) < len(diff_gdf):
    print(f"Difference map: plotting sample {len(diff_plot_gdf):,} / {len(diff_gdf):,} features")

norm_diff = TwoSlopeNorm(vmin=-cap, vcenter=0.0, vmax=cap)
cmap_diff = LinearSegmentedColormap.from_list('red_white_blue', ['#c81e1e', '#ffffff', '#1f78b4'], N=256)

fig, ax = plt.subplots(figsize=(11, 9))
ax.set_facecolor('#ffffff')
jamaica_boundary.boundary.plot(ax=ax, color='#bdbdbd', linewidth=0.45, zorder=1)

geom_type = diff_plot_gdf.geometry.geom_type.astype(str)
polys = diff_plot_gdf[geom_type.str.contains('Polygon', na=False)]
lines = diff_plot_gdf[geom_type.str.contains('LineString', na=False)]
points = diff_plot_gdf[geom_type.str.contains('Point', na=False)]

if not polys.empty:
    polys.plot(ax=ax, column='_plot_val', cmap=cmap_diff, norm=norm_diff, linewidth=0.10, edgecolor='none', alpha=0.9, zorder=2)
if not lines.empty:
    lines.plot(ax=ax, column='_plot_val', cmap=cmap_diff, norm=norm_diff, linewidth=0.85, alpha=0.95, zorder=3)
if not points.empty:
    points.plot(ax=ax, column='_plot_val', cmap=cmap_diff, norm=norm_diff, markersize=14, alpha=0.95, zorder=4)

sm = ScalarMappable(norm=norm_diff, cmap=cmap_diff)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label(f'Baseline EAD difference (Max - Min) ({diff_unit}), clipped at q=0.995')

ax.set_title('Baseline EAD difference map: Maximum minus Minimum scenario', fontsize=12)
ax.set_axis_off()

out_diff_png = comparison_out_dir / 'baseline_ead_difference_max_minus_min.png'
fig.savefig(out_diff_png, dpi=300, bbox_inches='tight')
print('Saved:', out_diff_png)
plt.show()


In [ ]:
# Quick comparison table (all-sector totals)

def format_usd_readable(value):
    abs_value = abs(float(value))
    sign = '-' if float(value) < 0 else ''
    if abs_value >= 1_000_000_000:
        return f"{sign}${abs_value / 1_000_000_000:,.2f} billion"
    if abs_value >= 1_000_000:
        return f"{sign}${abs_value / 1_000_000:,.2f} million"
    if abs_value >= 1_000:
        return f"{sign}${abs_value / 1_000:,.2f} thousand"
    return f"{sign}${abs_value:,.2f}"

min_total = float(baseline_min_gdf['EAD_Baseline_USD'].sum())
max_total = float(baseline_max_gdf['EAD_Baseline_USD'].sum())
diff_total = max_total - min_total
pct_diff = (100.0 * diff_total / min_total) if min_total > 0 else np.nan

comparison_df = pd.DataFrame([
    {'Metric': 'Baseline EAD total (minimum)', 'USD': min_total},
    {'Metric': 'Baseline EAD total (maximum)', 'USD': max_total},
    {'Metric': 'Difference (max - min)', 'USD': diff_total},
])
comparison_df['USD_Readable'] = comparison_df['USD'].apply(format_usd_readable)

pct_df = pd.DataFrame([{
    'Metric': 'Percent difference (max vs min)',
    'Percent': pct_diff,
    'Percent_Label': 'NA' if pd.isna(pct_diff) else f"{pct_diff:.2f}%"
}])

comparison_csv = comparison_out_dir / 'baseline_ead_min_max_total_comparison.csv'
comparison_df.to_csv(comparison_csv, index=False)

print('Baseline total comparison:')
display(comparison_df)
display(pct_df)
print('Saved:', comparison_csv)
